# Análisis de Spotify Global Music Dataset (2009–2025)

## Introducción

Este análisis explora el dataset global de Spotify con canciones y artistas 
de los últimos 16 años. El objetivo es responder las siguientes preguntas:

- ¿Qué géneros musicales dominan la plataforma?
- ¿Cómo evolucionó el contenido explícito a lo largo del tiempo?
- ¿Existe relación entre la popularidad de un artista y sus seguidores?

**Herramientas utilizadas:** Python · Pandas · Plotly  
**Dataset:** [Spotify Global Music Dataset — Kaggle](https://www.kaggle.com/datasets/wardabilal/spotify-global-music-dataset-20092025)

## 1. Importación de librerías

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.express as px # for visualization
import plotly.offline as py
import plotly.graph_objs as go
from plotly.figure_factory import create_table # for creating nice table

## 2. Carga del dataset

Se utiliza el archivo `spotify_data clean.csv` descargado desde Kaggle.
A continuación se visualizan las primeras filas para entender su estructura.

In [ ]:
import pandas as pd
df = pd.read_csv("spotify_data clean.csv")

In [ ]:
# A. Selección del dataframe
# Se selecciona el dataframe de kaggle con el nombre "spotify_data clean.csv" (https://www.kaggle.com/datasets/wardabilal/spotify-global-music-dataset-20092025) el cual cuenta con datos de artistas y canciones modernas mas escuchadas en este
# ultimo tiempo ( principalmente año 2025).

# A.1. Se importa el dataframe
import pandas as pd
df = pd.read_csv("spotify_data clean.csv")
df.head()

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


### Tipos de datos por columna

In [ ]:
# Conocer el tipo de atributos que el dataframe contiene
df.dtypes

track_id                  str
track_name                str
track_number            int64
track_popularity        int64
explicit                 bool
artist_name               str
artist_popularity       int64
artist_followers        int64
artist_genres             str
album_id                  str
album_name                str
album_release_date        str
album_total_tracks      int64
album_type                str
track_duration_min    float64
dtype: object

### Dimensiones del dataset

In [ ]:
# La cantidad de registros y atributos que el dataframe contiene
df.shape

(8582, 15)

## 3. Limpieza de datos

Se realizan los siguientes pasos:
- Conversión de `album_release_date` a formato datetime y extracción del año
- Revisión y tratamiento de valores nulos
- Creación de columna `main_genre` tomando el primer género del artista
- Eliminación de registros duplicados

In [ ]:
# A. 2. Limpieza y orden al dataset
# A.2.2. Se convierte el atributo album_release_date a tipo datetime
df['album_release_date'] = pd.to_datetime(df['album_release_date'], errors='coerce')
df['year'] = df['album_release_date'].dt.year

# A.2.3. Revisar los valores faltantes
df.isna().sum()

track_id                 0
track_name               0
track_number             0
track_popularity         0
explicit                 0
artist_name              3
artist_popularity        0
artist_followers         0
artist_genres         3361
album_id                 0
album_name               0
album_release_date       0
album_total_tracks       0
album_type               0
track_duration_min       0
year                     0
dtype: int64

In [ ]:
# Se observa que el atributo 'artis_genres' tiene mas de un valor, separados por una coma. Se decide crear una nueva columna 'main_genre' con el primer elemento de la columna
# 'artist_genre'. Posteriormente a los valores NaN se los cambia por 'Unknown.
df['main_genre'] = df['artist_genres'].str.split(',').str[0].fillna('Unknown')
df.isna().sum()


track_id                 0
track_name               0
track_number             0
track_popularity         0
explicit                 0
artist_name              3
artist_popularity        0
artist_followers         0
artist_genres         3361
album_id                 0
album_name               0
album_release_date       0
album_total_tracks       0
album_type               0
track_duration_min       0
year                     0
main_genre               0
dtype: int64

> Luego de la limpieza, los valores nulos fueron tratados correctamente. 
> La columna `main_genre` fue creada exitosamente a partir de `artist_genres`.

In [ ]:
# # A.2.4. Revisar y eliminar duplicados
df = df.drop_duplicates()

## 4. Visualizaciones

### 4.1 ¿Qué géneros musicales tienen más canciones en Spotify?

Se agrupa el dataset por `main_genre` y se cuentan las canciones por género.
Se muestran los **30 géneros con mayor presencia**, excluyendo entradas sin clasificar.

In [ ]:
# C. Creación de figuras

# C.1. Grafico de barras
# Se utilizará la categoria 'main_genre' para contar la cantidad de canciones por genero

# Cuenta por main_genre
df_count_main = df.groupby('main_genre').size().reset_index(name='count')
#Se ordena de mayor a menor
df_count_main = df_count_main.sort_values('count', ascending=False)
#Se Grafica
fig = px.bar(df_count_main, x='main_genre', y='count', height=600)
fig.show()
fig.write_image("generos_todos.png")

![Géneros todos](generos_todos.png)

> **Hallazgo:** El pop y sus subgéneros concentran la mayor cantidad de canciones, 
> seguidos por el hip-hop y el r&b.

In [ ]:
# Se observa que el genero unknown esta dominando la figura, por lo que se decide no tenerla en cuenta en cuenta y enfocarnos en los 30 génermos mas importantes.

# Cuenta por main_genre sin 'Unknown'
df_count_main = df_count_main[df_count_main['main_genre']!='Unknown']
#Se ordena de mayor a menor y se toman los primeros 30
df_top30 = df_count_main.sort_values('count', ascending=False).head(30)
#Se Grafica
fig = px.bar(
    df_top30,
    x='main_genre',
    y='count',
    text='count',
    height=600,
    title='Los 30 géneros musicales con mas canciones',
    labels={'main_genre': 'Género musical', 'album_type': 'Tipo de álbum', 'count': 'Cantidad de canciones'},
    )
fig.show()
fig.write_image("generos_top30.png")

![Top 30 géneros](generos_top30.png)

### 4.2 ¿Qué géneros tienen más contenido explícito?

Se analiza la proporción de canciones explícitas vs no explícitas 
dentro de los 30 géneros más representados.

In [ ]:
# Otra opción para el grafico de barras seria agrupar por cantidad de canciones identificadas como 'explicitas' por género musical.

top30_genres = df_top30['main_genre']
df_explicit = df[df['main_genre'].isin(top30_genres)]

df_grouped = (
    df_explicit.groupby(['main_genre', 'explicit'])
    .size()
    .reset_index(name='count')
)

df_grouped['explicit'] = df_grouped['explicit'].map({True: 'Sí', False: 'No'})

fig = px.bar(
    df_grouped,
    x='main_genre',
    y='count',
    color='explicit',
    barmode='group',
    title='Cantidad de canciones por género (agrupado por contenido explícito)',
    labels={
        'main_genre': 'Género musical',
        'count': 'Cantidad de canciones',
        'explicit': 'Contenido explícito'
    },
    height=600
)

fig.update_layout(
    xaxis_tickangle=45,
    title_x=0.5,
    legend_title="Explícito"
)

fig.show()
fig.write_image("generos_explicito.png")

![Contenido explícito por género](generos_explicito.png)

> **Hallazgo:** Géneros como el rap y el hip-hop presentan la mayor proporción 
> de contenido explícito, mientras que el pop y el indie muestran niveles más bajos.

### 4.3 ¿Cómo evolucionó la producción musical en Spotify a lo largo del tiempo?

Se analiza la cantidad de canciones lanzadas por año desde 2009 hasta 2025.

In [ ]:
# C.2. Series de tiempo
# Se realizará una serie temporal de la evolución en la creación de nuevas canciones a lo largo del tiempo
# Se utilizará la variable 'year' creada anteriormente.

# Contar canciones por año
df_year_count = df.groupby('year').size().reset_index(name='count')

fig = px.line(
    df_year_count,
    x='year',
    y='count',
    title='Cantidad de canciones lanzadas por año',
    labels={'year':'Año', 'count':'Cantidad de canciones'},
    markers=True,
    template='plotly_dark'
)

fig.update_layout(title_x=0.5)
fig.show()
fig.write_image("canciones_por_ano.png")


![Canciones por año](canciones_por_ano.png)

### 4.4 ¿Creció el contenido explícito con los años?

Se desagrega la evolución temporal separando canciones explícitas de no explícitas.

In [ ]:
# Podría mejorarse la grafica temporal y agrupar los lanzamientos en categorias para ver la evolución de las canciones explícitas vs las no explícitas.

df['explicit_label'] = df['explicit'].map({True: 'Sí', False: 'No'})

# Cantidad de canciones por año según explicit
df_year_explicit = (
    df.groupby(['year', 'explicit_label'])
    .size()
    .reset_index(name='count')
)

fig = px.line(
    df_year_explicit,
    x='year',
    y='count',
    color='explicit_label',
    markers=True,
    title='Lanzamientos por año según contenido explícito',
    labels={
        'year': 'Año',
        'count': 'Cantidad de canciones',
        'explicit_label': 'Explícito'
    },
    template='plotly_dark'
)

fig.update_layout(title_x=0.5)
fig.show()
fig.write_image("explicito_por_ano.png")


![Top 5 géneros](explicito_por_ano.png)

> **Hallazgo:** El contenido explícito creció sostenidamente desde 2015, 
> reflejando un cambio cultural en los géneros dominantes de la plataforma.

In [ ]:
# Una segunda opción a la gráfica temporal sería agrupar los lanzamientos en categorias para ver la evolución de las canciones de acuerdo al género musical.
# 1. Conteo total por género
df_count_main = df.groupby('main_genre').size().reset_index(name='count')
df_count_main = df_count_main[df_count_main['main_genre']!='Unknown']

# 2. Seleccionar top 5 géneros
top5_genres = df_count_main.sort_values('count', ascending=False).head(5)['main_genre']

# 3. Filtrar dataset solo con estos géneros
df_top5 = df[df['main_genre'].isin(top5_genres)]

# 4. Contar canciones por año y género
df_year_genre = (
    df_top5.groupby(['year', 'main_genre'])
    .size()
    .reset_index(name='count')
)

# 5. Gráfico temporal
fig = px.line(
    df_year_genre,
    x='year',
    y='count',
    color='main_genre',
    markers=True,
    title='Evolución de la cantidad de canciones lanzadas por año según género (Top 5)',
    labels={
        'year': 'Año',
        'count': 'Cantidad de canciones',
        'main_genre': 'Género musical'
    },
    template='plotly_dark',
    height=600
)

fig.update_layout(title_x=0.5)
fig.show()
fig.write_image("generos_top5.png")

![Top 5 géneros](generos_top5.png)

In [ ]:
# Se ve que es confuso para visualizar, se opta por crear una pequeña gráfica para cada genero usando el parametro 'facet_col'.
fig = px.line(
    df_year_genre,
    x='year',
    y='count',
    color='main_genre',
    facet_col='main_genre',
    facet_col_wrap=3,  # cantidad de columnas (ajústalo a gusto)
    markers=True,
    title='Evolución de la cantidad de canciones por año según género (Top 5 con facet)',
    labels={'year':'Año', 'count':'Cantidad de canciones', 'main_genre':'Género'},
    template='plotly_dark',
    height=700
)

fig.show()
fig.write_image("generos_top_5_facet.png")

![Top 5 géneros facet](generos_top_5_facet.png)

### 4.6 ¿Tienen más seguidores los artistas más populares?

Análisis multidimensional de los **20 artistas más populares** de Spotify:

| Dimensión | Variable |
|---|---|
| Eje X | Cantidad de seguidores (escala logarítmica) |
| Eje Y | Popularidad del artista (0–100) |
| Tamaño | Cantidad de canciones en el dataset |
| Color | Género musical principal |

In [ ]:
# C.3. Avanzada

# En cuanto a la gráfica avanzada, se elige hacer un scatter plot de los 20 artistas mas populares de Spotify, donde:
# Eje X: Cantidad de seguidores del artista en Spotify
# Eje Y: Popularidad del artista medido principalmente a partir de la cantidad y frecuencia de reproducciones recientes,
#         la popularidad de las canciones principales del artista y métricas de engagement del usuario (guardados, saltos,
#         repeticiones, inclusión en playlists).
# size: Cada punto representa a un artista dentro de los 20 mas populares y el tamaño de la burbuja, representa la cantidad de
#       canciones escuchadas que tiene el artista en la plataforma al momento del dataset.
# Labels: Género muscial de los artistas


# Seleccionar los 20 artistas con mayor popularidad
artist_pop = df[['artist_name', 'artist_popularity']].drop_duplicates()

top20_popular_artists = (
    artist_pop
    .sort_values('artist_popularity', ascending=False)
    .head(20)['artist_name']
)

# Filtrar el dataset para quedarnos solo con esos artistas
df_top20 = df[df['artist_name'].isin(top20_popular_artists)]

# Calcular cuántas canciones tiene cada uno (para usarlo como size)
song_count = (
    df_top20.groupby('artist_name')
            .size()
            .reset_index(name='song_count')
)

# Unir la columna song_count al df_top20
df_top20 = df_top20.merge(song_count, on='artist_name', how='left')

# Agregar main_genre
df_top20['main_genre'] = df_top20['artist_genres'].str.split(',').str[0].fillna('Unknown')
df_top20 = df_top20[df_top20['main_genre'] != 'Unknown']

# Crear el Scatter Plot con size=cantidad de canciones
fig = px.scatter(
    df_top20,
    x='artist_followers',
    y='artist_popularity',
    size='song_count',
    color='main_genre',
    hover_name='artist_name',
    hover_data={
        'song_count': True,
        'track_name': True,
        'track_popularity': True,
        'artist_followers': True,
        'artist_popularity': True,
        'main_genre': True
    },
    title='Top 20 artistas más populares: seguidores vs popularidad (tamaño = cantidad de canciones)',
    labels={
        'artist_followers': 'Seguidores del artista',
        'artist_popularity': 'Popularidad del artista (0: poco popular, 100: muy popular)',
        'main_genre': 'Género musical',
        'song_count': 'Cantidad de canciones'
    },
    size_max=60,
    template='plotly_dark',
    height=750
)

fig.update_layout(
    title_x=0.5,
    legend_title='Género musical',
    xaxis=dict(type='log')
)

fig.show()
fig.write_image("artistas_top20.png")

![Top 20 artistas](artistas_top20.png)

## 5. Conclusiones

- El **pop** domina ampliamente Spotify en volumen de canciones
- El contenido **explícito creció sostenidamente desde 2015**, liderado por rap y hip-hop
- **No existe una correlación lineal perfecta** entre seguidores y popularidad: 
  artistas con menos seguidores pueden tener alta popularidad por reproducciones recientes
- La producción musical registrada en Spotify muestra un **pico entre 2020 y 2023**

---
*Análisis realizado por Juan Bautista Acuña — [GitHub](https://github.com/bautistaacuna)*